# Домашнее задание по генерации речи

## Few-shot voice cloning модели XTTS


Cтатья: [XTTS: a Massively Multilingual Zero-Shot Text-to-Speech Model](https://arxiv.org/abs/2406.04904).

**github**: https://github.com/coqui-ai/TTS

**документация:** https://docs.coqui.ai/en/dev/models/xtts.html

**demo**: https://edresson.github.io/XTTS/

На семинаре были примеры спикеров, на которых zero-shot генерация по одной референсной записи показывала не очень высокий уровень похожести голоса. Альтернативным сценарием явялется дообучение модели на данных такого диктора, которое и предлагается выполнить в этом домашнем задании.

## 1. Настройка окружения

Установите необходимые пакеты, следуя документации (установка через pip либо скачивание репозитория с github).

In [1]:
!pip install -U pip
!pip install -U coqui-tts librosa soundfile pydub
!apt-get update -qq
!apt-get install -y -qq ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [40]:
import torch
import torch.nn.functional as F
import dataclasses

import gdown
from pathlib import Path
import tarfile

from pathlib import Path
import shutil
import subprocess
import soundfile as sf
import librosa
import numpy as np

from dataclasses import dataclass
from TTS.api import TTS
from IPython.display import Audio, display

import re
import os
import gc

import kagglehub

from trainer import Trainer, TrainerArgs

from TTS.config.shared_configs import BaseDatasetConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.layers.xtts.trainer.gpt_trainer import (
    GPTArgs,
    GPTTrainer,
    GPTTrainerConfig,
)
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

try:
    from TTS.tts.layers.xtts.trainer.gpt_trainer import XttsAudioConfig
except ImportError:
    from TTS.tts.configs.xtts_config import XttsAudioConfig

from TTS.utils.manage import ModelManager
from transformers import WavLMForXVector, Wav2Vec2FeatureExtractor
import torchaudio

In [3]:
@dataclass(slots=True, frozen=True)
class Config:
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    data_path: Path = Path("/kaggle/working")

config = Config()

## 2. Сбор и подготовка данных



### Загрузка данных
Используется датасет [Dmitri 1.0](https://mozilladatacollective.com/datasets/cmiup9hrx01hsmf074vi5iqgz)


In [4]:
url = "https://drive.google.com/file/d/1ne4dbwLssnl6lA5JymPPI2hGl-7COxgr/view?usp=sharing"
config.data_path.mkdir(parents=True, exist_ok=True)

data_path = config.data_path / "dmitri-1-0-fd227527.tar.gz"

gdown.download(
    url=url,
    output=str(data_path),
    quiet=False,
    fuzzy=True,
)

with tarfile.open(data_path, "r:gz") as tar:
    tar.extractall(config.data_path)

Downloading...
From (original): https://drive.google.com/uc?id=1ne4dbwLssnl6lA5JymPPI2hGl-7COxgr
From (redirected): https://drive.google.com/uc?id=1ne4dbwLssnl6lA5JymPPI2hGl-7COxgr&confirm=t&uuid=70434ee7-8917-423d-91c3-ca091d59b8a7
To: /kaggle/working/dmitri-1-0-fd227527.tar.gz
100%|██████████| 101M/101M [00:00<00:00, 105MB/s]  
/tmp/ipykernel_10843/1423324688.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(config.data_path)


### Подготовка данных

In [5]:
root = config.data_path / Path("ru-RU/3000000001_3000000300_Chat")

audio_files = []
for ext in ["*.webm"]:
    audio_files.extend(root.rglob(ext))

print("Audio files:", len(audio_files))

metadata_files = []
for ext in ["*.txt"]:
    metadata_files.extend(root.rglob(ext))

print("\nMetadata candidates:", len(metadata_files))
for p in metadata_files[:10]:
    print(p)

Audio files: 300

Metadata candidates: 300
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000032.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000012.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000068.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000055.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000162.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000089.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000218.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000006.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000220.txt
/kaggle/working/ru-RU/3000000001_3000000300_Chat/3000000003.txt


In [6]:
TARGET_SR = 24000
MAX_SECONDS = 10 * 60  # 10 минут
MIN_UTT_SECONDS = 2.0
MAX_UTT_SECONDS = 15.0

src_files = sorted(audio_files)

out_dir = config.data_path / Path("dmitri_subset/wavs")
out_dir.mkdir(parents=True, exist_ok=True)

selected = []
total_seconds = 0.0


def convert_to_wav(src: Path, dst: Path, sr: int = TARGET_SR) -> None:
    # ffmpeg хорошо конвертирует WEBM -> WAV
    cmd = [
        "ffmpeg",
        "-y",
        "-i", str(src),
        "-ac", "1",
        "-ar", str(sr),
        "-sample_fmt", "s16",
        str(dst),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def get_duration_seconds(path: Path) -> float:
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate)


for i, src in enumerate(src_files):
    tmp_wav = out_dir / f"utt_{i:04d}.wav"

    try:
        convert_to_wav(src, tmp_wav, TARGET_SR)
        duration = get_duration_seconds(tmp_wav)

        if duration < MIN_UTT_SECONDS or duration > MAX_UTT_SECONDS:
            tmp_wav.unlink(missing_ok=True)
            continue

        selected.append(tmp_wav)
        total_seconds += duration

        if total_seconds >= MAX_SECONDS:
            break

    except Exception as e:
        print("Skip:", src, e)
        tmp_wav.unlink(missing_ok=True)

print(f"Selected files: {len(selected)}")
print(f"Total duration: {total_seconds / 60:.2f} min")

for p in selected[:10]:
    print(p)

Selected files: 143
Total duration: 10.01 min
/kaggle/working/dmitri_subset/wavs/utt_0000.wav
/kaggle/working/dmitri_subset/wavs/utt_0001.wav
/kaggle/working/dmitri_subset/wavs/utt_0002.wav
/kaggle/working/dmitri_subset/wavs/utt_0003.wav
/kaggle/working/dmitri_subset/wavs/utt_0004.wav
/kaggle/working/dmitri_subset/wavs/utt_0005.wav
/kaggle/working/dmitri_subset/wavs/utt_0006.wav
/kaggle/working/dmitri_subset/wavs/utt_0007.wav
/kaggle/working/dmitri_subset/wavs/utt_0008.wav
/kaggle/working/dmitri_subset/wavs/utt_0009.wav


### Zero-shot baseline

In [32]:
%env COQUI_TOS_AGREED=1

env: COQUI_TOS_AGREED=1


In [10]:
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(config.device)

tts.tts_to_file(
    text="Привет. Это тестовая генерация речи до дообучения модели.",
    speaker_wav= Path(config.data_path) / "dmitri_subset/wavs/utt_0000.wav",
    language="ru",
    file_path="outputs/zero_shot_baseline.wav",
)

100%|██████████| 1.87G/1.87G [00:19<00:00, 96.8MiB/s]
4.37kiB [00:00, 6.20MiB/s]
361kiB [00:00, 31.6MiB/s]
100%|██████████| 32.0/32.0 [00:00<00:00, 47.7kiB/s]
100%|██████████| 7.75M/7.75M [00:00<00:00, 60.4MiB/s]


'outputs/zero_shot_baseline.wav'

In [14]:
display(Audio(Path(config.data_path) / "dmitri_subset/wavs/utt_0000.wav"))

In [15]:
display(Audio("outputs/zero_shot_baseline.wav"))

Zero-shot генерация по одной референсной записи дала недостаточную похожесть: тембр/интонация/ритм отличались от исходного диктора.

## 3. Дообучение



In [7]:
SPEAKER_NAME = "dmitri"
LANGUAGE = "ru"

FT_DATA_ROOT = config.data_path / Path("xtts_finetune_dmitri")
FT_1MIN_ROOT = FT_DATA_ROOT / "dmitri_1min"
FT_ALL_ROOT = FT_DATA_ROOT / "dmitri_all"


def clean_text(text: str) -> str:
    text = text.replace("\ufeff", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def find_transcript_for_audio(src_audio: Path) -> str | None:
    """
    Пытаемся найти .txt рядом с исходным .webm.
    Для Dmitri чаще всего transcript лежит рядом или имеет тот же stem.
    """
    candidates = [
        src_audio.with_suffix(".txt"),
        src_audio.parent / f"{src_audio.stem}.txt",
    ]

    for candidate in candidates:
        if candidate.exists():
            text = clean_text(candidate.read_text(encoding="utf-8", errors="ignore"))
            if text:
                return text

    txt_files = sorted(src_audio.parent.glob("*.txt"))
    if len(txt_files) == 1:
        text = clean_text(txt_files[0].read_text(encoding="utf-8", errors="ignore"))
        if text:
            return text

    return None


def build_xtts_dataset(
    src_files: list[Path],
    dataset_root: Path,
    max_seconds: float,
    target_sr: int = TARGET_SR,
    min_utt_seconds: float = MIN_UTT_SECONDS,
    max_utt_seconds: float = MAX_UTT_SECONDS,
) -> Path:
    """
    Создает датасет в LJSpeech-like формате:

    dataset_root/
      metadata.txt
      wavs/
        utt_0000.wav
        ...

    metadata.txt:
      utt_0000|текст|текст
    """
    wavs_dir = dataset_root / "wavs"

    if dataset_root.exists():
        shutil.rmtree(dataset_root)

    wavs_dir.mkdir(parents=True, exist_ok=True)

    metadata_lines = []
    total_seconds = 0.0
    used = 0

    for src in sorted(src_files):
        text = find_transcript_for_audio(src)
        if not text:
            print(f"Skip without transcript: {src}")
            continue

        dst_wav = wavs_dir / f"utt_{used:04d}.wav"

        try:
            convert_to_wav(src, dst_wav, target_sr)
            duration = get_duration_seconds(dst_wav)

            if duration < min_utt_seconds or duration > max_utt_seconds:
                dst_wav.unlink(missing_ok=True)
                continue

            # LJSpeech formatter ожидает имя без .wav
            stem = dst_wav.stem
            metadata_lines.append(f"{stem}|{text}|{text}")

            total_seconds += duration
            used += 1

            if total_seconds >= max_seconds:
                break

        except Exception as e:
            print(f"Skip corrupted/problem file: {src} | {e}")
            dst_wav.unlink(missing_ok=True)

    metadata_path = dataset_root / "metadata.txt"
    metadata_path.write_text("\n".join(metadata_lines), encoding="utf-8")

    print(f"Dataset: {dataset_root}")
    print(f"Utterances: {used}")
    print(f"Duration: {total_seconds / 60:.2f} min")
    print(f"Metadata: {metadata_path}")

    return metadata_path

In [8]:
metadata_1min = build_xtts_dataset(
    src_files=src_files,
    dataset_root=FT_1MIN_ROOT,
    max_seconds=60.0,
)

metadata_all = build_xtts_dataset(
    src_files=src_files,
    dataset_root=FT_ALL_ROOT,
    max_seconds=10 * 60.0,
)

Dataset: /kaggle/working/xtts_finetune_dmitri/dmitri_1min
Utterances: 14
Duration: 1.05 min
Metadata: /kaggle/working/xtts_finetune_dmitri/dmitri_1min/metadata.txt
Dataset: /kaggle/working/xtts_finetune_dmitri/dmitri_all
Utterances: 143
Duration: 10.01 min
Metadata: /kaggle/working/xtts_finetune_dmitri/dmitri_all/metadata.txt


### Сохранение модели в Kaggle 

Сохраянем модели отдельно потому что не кагле на диске не хватает места для хранения модели и логов обучения.

In [9]:
def build_kaggle_model_info_from_ckpt(ft_ckpt: str | Path) -> tuple[Path, str]:
    ft_ckpt = Path(ft_ckpt).resolve()

    if not ft_ckpt.exists():
        raise FileNotFoundError(ft_ckpt)

    local_model_dir = ft_ckpt.parent

    # Например:
    # XTTS_Dmitri_1min_FT-April-28-2026_12+28PM-0000000
    # -> XTTS_Dmitri_1min_FT-April-28-2026_12
    model_slug = re.sub(r"\+\d{2}[AP]M-\d+$", "", local_model_dir.name)

    return local_model_dir, model_slug
    
def link_file(src: Path, dst: Path) -> None:
    if dst.exists():
        dst.unlink()

    try:
        os.link(src, dst)  # без копирования, если тот же filesystem
    except OSError:
        dst.symlink_to(src)  # fallback: symlink


def prepare_minimal_xtts_model_dir_no_copy(
    checkpoint_path: str | Path,
    output_dir: str | Path,
) -> Path:
    checkpoint_path = Path(checkpoint_path).resolve()
    output_dir = Path(output_dir)

    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    config_path = find_config_for_checkpoint(checkpoint_path).resolve()

    output_dir.mkdir(parents=True, exist_ok=True)

    link_file(checkpoint_path, output_dir / "best_model.pth")
    link_file(config_path, output_dir / "config.json")

    print("Minimal XTTS model dir:", output_dir)
    print("Checkpoint link:", output_dir / "best_model.pth", "->", checkpoint_path)
    print("Config link:", output_dir / "config.json", "->", config_path)

    return output_dir

In [10]:
def find_file_in_tree_or_parents(path: str | Path, filename: str) -> Path:
    path = Path(path).resolve()
    search_root = path if path.is_dir() else path.parent

    found = sorted(search_root.rglob(filename))
    if found:
        return found[0].resolve()

    for parent in [search_root, *search_root.parents]:
        candidate = parent / filename
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(f"{filename} не найден рядом с {path}")
    
def find_best_checkpoint(path: str | Path) -> Path:
    path = Path(path)

    if path.is_file() and path.suffix == ".pth":
        return path.resolve()

    if not path.exists():
        raise FileNotFoundError(f"Путь не найден: {path}")

    checkpoints = sorted(path.rglob("best_model.pth"))

    if not checkpoints:
        checkpoints = sorted(path.rglob("*.pth"))

    if not checkpoints:
        raise FileNotFoundError(f"Checkpoint не найден в {path}")

    return max(checkpoints, key=lambda p: p.stat().st_mtime).resolve()


def find_config_for_checkpoint(checkpoint_path: str | Path) -> Path:
    checkpoint_path = Path(checkpoint_path).resolve()

    for parent in [checkpoint_path.parent, *checkpoint_path.parents]:
        config_path = parent / "config.json"
        if config_path.exists():
            return config_path.resolve()

    raise FileNotFoundError(
        f"config.json не найден рядом с checkpoint: {checkpoint_path}"
    )


def build_kaggle_model_info_from_ckpt(ft_ckpt: str | Path) -> tuple[Path, str]:
    ft_ckpt = Path(ft_ckpt).resolve()

    if not ft_ckpt.exists():
        raise FileNotFoundError(ft_ckpt)

    local_model_dir = ft_ckpt.parent

    model_slug = re.sub(
        r"\+\d{2}[AP]M-\d+$",
        "",
        local_model_dir.name,
    )

    return local_model_dir, model_slug


def link_file_no_copy(src: Path, dst: Path) -> None:
    src = src.resolve()

    if dst.exists() or dst.is_symlink():
        dst.unlink()

    try:
        os.link(src, dst)       # hardlink: не занимает место повторно
    except OSError:
        dst.symlink_to(src)     # fallback


def upload_minimal_xtts_model_from_ckpt_no_copy(
    ft_ckpt: str | Path,
    owner: str = "twogenies",
    variation_slug: str = "XTTS",
    version_notes: str = "Minimal XTTS checkpoint + config only",
) -> Path:
    local_model_dir, model_slug = build_kaggle_model_info_from_ckpt(ft_ckpt)

    checkpoint_path = find_best_checkpoint(local_model_dir)
    config_path = find_config_for_checkpoint(checkpoint_path)
    vocab_path = find_file_in_tree_or_parents(local_model_dir, "vocab.json")

    minimal_dir = local_model_dir.parent / f"{local_model_dir.name}_minimal"
    minimal_dir.mkdir(parents=True, exist_ok=True)

    link_file_no_copy(checkpoint_path, minimal_dir / "best_model.pth")
    link_file_no_copy(config_path, minimal_dir / "config.json")
    link_file_no_copy(vocab_path, minimal_dir / "vocab.json")

    print("Uploading minimal XTTS model:")
    print("LOCAL_MODEL_DIR:", local_model_dir)
    print("MODEL_SLUG:", model_slug)
    print("minimal_dir:", minimal_dir)
    print("checkpoint:", checkpoint_path)
    print("config:", config_path)
    print("vocab:", vocab_path)

    kagglehub.model_upload(
        handle=f"{owner}/{model_slug}/keras/{variation_slug}",
        local_model_dir=str(minimal_dir),
        version_notes=version_notes,
    )

    return minimal_dir

### Обучение

In [11]:
def clear_cuda() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def download_xtts_v2_files(out_path: Path) -> dict[str, str]:
    checkpoints_dir = out_path / "XTTS_v2_original_model_files"
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    links = {
        "dvae_checkpoint": "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/dvae.pth",
        "mel_norm_file": "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/mel_stats.pth",
        "tokenizer_file": "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/vocab.json",
        "xtts_checkpoint": "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/model.pth",
    }

    paths = {
        key: str(checkpoints_dir / Path(url).name)
        for key, url in links.items()
    }

    missing_links = [
        url for key, url in links.items()
        if not Path(paths[key]).exists()
    ]

    if missing_links:
        ModelManager._download_model_files(
            missing_links,
            str(checkpoints_dir),
            progress_bar=True,
        )

    return paths


def train_xtts_gpt(
    dataset_root: Path,
    run_name: str,
    output_root: Path,
    epochs: int,
    batch_size: int = 2,
    grad_accum_steps: int = 32,
    lr: float = 5e-6,
) -> Path:
    """
    Fine-tuning XTTS-v2 GPT-компонента.

    Возвращает путь к best_model.pth.
    """
    clear_cuda()

    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    xtts_files = download_xtts_v2_files(output_root)

    dataset_config = BaseDatasetConfig(
        formatter="ljspeech",
        dataset_name=run_name,
        path=str(dataset_root),
        meta_file_train="metadata.txt",
        language=LANGUAGE,
    )

    model_args = GPTArgs(
        max_conditioning_length=132300,   # около 6 сек при 22050
        min_conditioning_length=66150,    # около 3 сек при 22050
        debug_loading_failures=True,
        max_wav_length=255995,            # около 11.6 сек
        max_text_length=200,
        mel_norm_file=xtts_files["mel_norm_file"],
        dvae_checkpoint=xtts_files["dvae_checkpoint"],
        xtts_checkpoint=xtts_files["xtts_checkpoint"],
        tokenizer_file=xtts_files["tokenizer_file"],
        gpt_num_audio_tokens=1026,
        gpt_start_audio_token=1024,
        gpt_stop_audio_token=1025,
        gpt_use_masking_gt_prompt_approach=True,
        gpt_use_perceiver_resampler=True,
    )

    audio_config = XttsAudioConfig(
        sample_rate=22050,
        dvae_sample_rate=22050,
        output_sample_rate=24000,
    )

    speaker_ref = str(next((dataset_root / "wavs").glob("*.wav")))

    train_config = GPTTrainerConfig(
        output_path=str(output_root),
        model_args=model_args,
        run_name=run_name,
        project_name="XTTS_Dmitri_FT",
        run_description=f"XTTS-v2 GPT fine-tuning on {dataset_root}",
        dashboard_logger="tensorboard",
        logger_uri=None,
        audio=audio_config,

        batch_size=batch_size,
        eval_batch_size=batch_size,
        batch_group_size=8,
        num_loader_workers=2,

        epochs=epochs,
        print_step=10,
        plot_step=50,
        log_model_step=100,
        save_step=1000,
        save_n_checkpoints=1,
        save_checkpoints=True,
        save_all_best = False,
        print_eval=True,

        optimizer="AdamW",
        optimizer_wd_only_on_weights=True,
        optimizer_params={
            "betas": [0.9, 0.96],
            "eps": 1e-8,
            "weight_decay": 1e-2,
        },
        lr=lr,
        lr_scheduler="MultiStepLR",
        lr_scheduler_params={
            "milestones": [1000, 2000, 4000],
            "gamma": 0.5,
            "last_epoch": -1,
        },

        test_sentences=[
            {
                "text": "Привет. Это проверка голоса после дообучения модели.",
                "speaker_wav": [speaker_ref],
                "language": LANGUAGE,
            },
            {
                "text": "Сегодня мы сравниваем качество zero-shot и few-shot синтеза речи.",
                "speaker_wav": [speaker_ref],
                "language": LANGUAGE,
            },
        ],
    )

    model = GPTTrainer.init_from_config(train_config)

    train_samples, eval_samples = load_tts_samples(
        [dataset_config],
        eval_split=True,
        eval_split_max_size=min(16, max(1, len(list((dataset_root / "wavs").glob("*.wav"))) // 5)),
        eval_split_size=0.1,
    )

    trainer = Trainer(
        TrainerArgs(
            restore_path=None,
            skip_train_epoch=False,
            start_with_eval=False,
            grad_accum_steps=grad_accum_steps,
        ),
        train_config,
        output_path=str(output_root),
        model=model,
        train_samples=train_samples,
        eval_samples=eval_samples,
        gpu=0,
    )

    trainer.fit()

    candidates = sorted(output_root.rglob("best_model.pth"))
    if not candidates:
        candidates = sorted(output_root.rglob("*.pth"))

    if not candidates:
        raise RuntimeError(f"Checkpoint not found {output_root}")

    best_ckpt = candidates[-1]
    print(f"Best checkpoint: {best_ckpt}")

    return best_ckpt

In [68]:
!rm -rf /kaggle/working/xtts_dmitri_1min
!rm -rf /kaggle/working/xtts_dmitri_all
clear_cuda()

In [12]:
output_root = Path("/kaggle/working/xtts_dmitri_1min")

ft_1min_ckpt = train_xtts_gpt(
    dataset_root=FT_1MIN_ROOT,
    run_name="XTTS_Dmitri_1min_FT",
    output_root=output_root,
    epochs=10,
    batch_size=4,
    grad_accum_steps=32,
    lr=5e-6,
)

print("1-minute fine-tuned checkpoint:", ft_1min_ckpt)


100%|██████████| 211M/211M [00:01<00:00, 107MiB/s]  
100%|██████████| 1.07k/1.07k [00:00<00:00, 3.67MiB/s]
361kiB [00:00, 30.7MiB/s]
100%|██████████| 1.87G/1.87G [00:17<00:00, 109MiB/s] 
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 2
 | > Num. of CPUs: 4
 | > Num. of Torch Threads: 1
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
2026-04-28 16:56:49.540259: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has a

Best checkpoint: /kaggle/working/xtts_dmitri_1min/XTTS_Dmitri_1min_FT-April-28-2026_04+56PM-0000000/best_model.pth
1-minute fine-tuned checkpoint: /kaggle/working/xtts_dmitri_1min/XTTS_Dmitri_1min_FT-April-28-2026_04+56PM-0000000/best_model.pth


In [27]:
LOCAL_MODEL_DIR = "/kaggle/working/xtts_dmitri_1min/XTTS_Dmitri_1min_FT-April-28-2026_04+56PM-0000000"
minimal_model_dir = upload_minimal_xtts_model_from_ckpt_no_copy(ft_ckpt=LOCAL_MODEL_DIR)

Uploading minimal XTTS model:
LOCAL_MODEL_DIR: /kaggle/working/xtts_dmitri_1min
MODEL_SLUG: xtts_dmitri_1min
minimal_dir: /kaggle/working/xtts_dmitri_1min_minimal
checkpoint: /kaggle/working/xtts_dmitri_1min/XTTS_Dmitri_1min_FT-April-28-2026_04+56PM-0000000/best_model.pth
config: /kaggle/working/xtts_dmitri_1min/XTTS_Dmitri_1min_FT-April-28-2026_04+56PM-0000000/config.json
vocab: /kaggle/working/xtts_dmitri_1min/XTTS_v2_original_model_files/vocab.json
Uploading Model https://api.kaggle.com/models/twogenies/xtts_dmitri_1min/keras/XTTS ...
Model 'xtts_dmitri_1min' does not exist or access is forbidden for user 'twogenies'. Creating or handling Model...
Model 'xtts_dmitri_1min' Created.
Starting upload for file /kaggle/working/xtts_dmitri_1min_minimal/best_model.pth


Uploading: 100%|██████████| 5.61G/5.61G [01:06<00:00, 84.0MB/s]

Upload successful: /kaggle/working/xtts_dmitri_1min_minimal/best_model.pth (5GB)
Starting upload for file /kaggle/working/xtts_dmitri_1min_minimal/config.json



Uploading: 100%|██████████| 6.55k/6.55k [00:00<00:00, 16.9kB/s]

Upload successful: /kaggle/working/xtts_dmitri_1min_minimal/config.json (6KB)
Starting upload for file /kaggle/working/xtts_dmitri_1min_minimal/vocab.json



Uploading: 100%|██████████| 361k/361k [00:00<00:00, 896kB/s]

Upload successful: /kaggle/working/xtts_dmitri_1min_minimal/vocab.json (353KB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/twogenies/xtts_dmitri_1min/keras/XTTS


In [43]:
clear_cuda()

In [12]:
output_root = Path("/kaggle/working/xtts_dmitri_all")

ft_all_ckpt = train_xtts_gpt(
    dataset_root=FT_ALL_ROOT,
    run_name="XTTS_Dmitri_All_FT",
    output_root=output_root,
    epochs=10,
    batch_size=4,
    grad_accum_steps=32,
    lr=5e-6,
)

print("All-data fine-tuned checkpoint:", ft_all_ckpt)

100%|██████████| 211M/211M [00:02<00:00, 95.8MiB/s] 
100%|██████████| 1.07k/1.07k [00:00<00:00, 1.81MiB/s]
361kiB [00:00, 27.0MiB/s]
100%|██████████| 1.87G/1.87G [00:18<00:00, 99.9MiB/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 2
 | > Num. of CPUs: 4
 | > Num. of Torch Threads: 1
 | > Torch seed: 1
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
2026-04-28 17:31:41.031424: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has a

Best checkpoint: /kaggle/working/xtts_dmitri_all/XTTS_Dmitri_All_FT-April-28-2026_05+31PM-0000000/best_model.pth
All-data fine-tuned checkpoint: /kaggle/working/xtts_dmitri_all/XTTS_Dmitri_All_FT-April-28-2026_05+31PM-0000000/best_model.pth


In [14]:
LOCAL_MODEL_DIR = "/kaggle/working/xtts_dmitri_all/XTTS_Dmitri_All_FT-April-28-2026_05+31PM-0000000"
minimal_model_all_dir = upload_minimal_xtts_model_from_ckpt_no_copy(ft_ckpt=LOCAL_MODEL_DIR)

Uploading minimal XTTS model:
LOCAL_MODEL_DIR: /kaggle/working/xtts_dmitri_all
MODEL_SLUG: xtts_dmitri_all
minimal_dir: /kaggle/working/xtts_dmitri_all_minimal
checkpoint: /kaggle/working/xtts_dmitri_all/XTTS_Dmitri_All_FT-April-28-2026_05+31PM-0000000/best_model.pth
config: /kaggle/working/xtts_dmitri_all/XTTS_Dmitri_All_FT-April-28-2026_05+31PM-0000000/config.json
vocab: /kaggle/working/xtts_dmitri_all/XTTS_v2_original_model_files/vocab.json
Uploading Model https://api.kaggle.com/models/twogenies/xtts_dmitri_all/keras/XTTS ...
Model 'xtts_dmitri_all' does not exist or access is forbidden for user 'twogenies'. Creating or handling Model...
Model 'xtts_dmitri_all' Created.
Starting upload for file /kaggle/working/xtts_dmitri_all_minimal/best_model.pth


Uploading: 100%|██████████| 5.61G/5.61G [01:10<00:00, 80.0MB/s]

Upload successful: /kaggle/working/xtts_dmitri_all_minimal/best_model.pth (5GB)
Starting upload for file /kaggle/working/xtts_dmitri_all_minimal/config.json



Uploading: 100%|██████████| 6.54k/6.54k [00:00<00:00, 16.5kB/s]

Upload successful: /kaggle/working/xtts_dmitri_all_minimal/config.json (6KB)
Starting upload for file /kaggle/working/xtts_dmitri_all_minimal/vocab.json



Uploading: 100%|██████████| 361k/361k [00:00<00:00, 928kB/s]

Upload successful: /kaggle/working/xtts_dmitri_all_minimal/vocab.json (353KB)


Your model instance has been created.
Files are being processed...
See at: https://api.kaggle.com/models/twogenies/xtts_dmitri_all/keras/XTTS


## 4. Оценка результатов

**4.1.** Для каждой из 3х моделей (zero-shot, дообученная на 1 минуте, дообученная на всех данных) сгенерируйте 10 предложени

In [20]:
class FineTunedXTTS:
    def __init__(
        self,
        checkpoint_path: str | Path,
        config_path: str | Path,
        device: str,
    ):
        self.checkpoint_path = Path(checkpoint_path)
        self.config_path = Path(config_path)
        self.device = device

        self.config = XttsConfig()
        self.config.load_json(str(self.config_path))

        vocab_path = self.config_path.parent / "vocab.json"
        self.config.model_args.tokenizer_file = str(vocab_path)


        self.model = Xtts.init_from_config(self.config)

        self.model.load_checkpoint(
            self.config,
            checkpoint_path=str(self.checkpoint_path),
            eval=True,
        )

        self.model.to(device)
        self.model.eval()

    @torch.inference_mode()
    def tts_to_file(
        self,
        text: str,
        speaker_wav: str | Path,
        language: str,
        file_path: str | Path,
    ) -> None:
        speaker_wav = str(speaker_wav)
        file_path = str(file_path)

        outputs = self.model.synthesize(
            text=text,
            config=self.config,
            speaker_wav=speaker_wav,
            language=language,
        )

        wav = outputs["wav"]

        import soundfile as sf

        sf.write(
            file_path,
            wav,
            self.config.audio.output_sample_rate,
        )


def find_best_checkpoint(path: str | Path) -> Path:
    path = Path(path)

    if path.is_file() and path.suffix == ".pth":
        print("Using checkpoint:", path)
        return path

    if not path.exists():
        raise FileNotFoundError(f"Путь не найден: {path}")

    preferred_names = [
        "best_model.pth",
        "best_model_40.pth",
        "checkpoint.pth",
        "model.pth",
    ]

    for name in preferred_names:
        found = sorted(path.rglob(name))
        if found:
            checkpoint = max(found, key=lambda p: p.stat().st_mtime)
            print("Found checkpoint:", checkpoint)
            return checkpoint

    checkpoints = sorted(path.rglob("*.pth"))

    if not checkpoints:
        raise FileNotFoundError(
            f"Checkpoint .pth не найден в {path}. "
            "Проверь структуру через: !find /kaggle/input -name '*.pth'"
        )

    checkpoint = max(checkpoints, key=lambda p: p.stat().st_mtime)
    print("Found checkpoint:", checkpoint)
    return checkpoint


def find_config_for_checkpoint(checkpoint_path: str | Path) -> Path:
    checkpoint_path = Path(checkpoint_path)

    for parent in [checkpoint_path.parent, *checkpoint_path.parents]:
        config_path = parent / "config.json"
        if config_path.exists():
            print("Found config:", config_path)
            return config_path

    raise FileNotFoundError(
        f"config.json не найден рядом с checkpoint: {checkpoint_path}. "
        "Проверь командой: !find /kaggle/input -name 'config.json'"
    )


def load_finetuned_tts_from_kaggle(
    kaggle_model_dir: str | Path,
    device: str,
) -> FineTunedXTTS:
    kaggle_model_dir = Path(kaggle_model_dir)

    checkpoint_path = find_best_checkpoint(kaggle_model_dir)
    config_path = find_config_for_checkpoint(checkpoint_path)
    vocab_path = config_path.parent / "vocab.json"

    print("Loading fine-tuned XTTS from Kaggle:")
    print("model dir:", kaggle_model_dir)
    print("checkpoint:", checkpoint_path)
    print("config:", config_path)
    print("vocab:", vocab_path)

    return FineTunedXTTS(
        checkpoint_path=checkpoint_path,
        config_path=config_path,
        device=device,
    )

In [22]:
EVAL_OUTPUT_DIR = Path("outputs/eval_4_1")
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_SENTENCES = [
    "Привет. Это первая тестовая фраза для проверки синтеза речи.",
    "Сегодня мы сравниваем качество разных моделей клонирования голоса.",
    "Мне нужно понять, насколько голос похож на исходного диктора.",
    "Эта запись используется только для оценки качества генерации.",
    "Модель должна сохранять тембр, ритм и интонацию говорящего.",
    "Русская речь должна звучать естественно и без заметных артефактов.",
    "После дообучения голос может стать более похожим на целевого диктора.",
    "Важно проверить не только качество речи, но и сходство голоса.",
    "Короткое дообучение иногда помогает улучшить zero-shot результат.",
    "Теперь сгенерируем несколько примеров для дальнейшей оценки.",
]


def generate_10_samples(
    model,
    model_name: str,
    speaker_wav: str | Path,
    sentences: list[str],
    output_dir: Path,
    language: str = "ru",
) -> list[Path]:
    model_output_dir = output_dir / model_name
    model_output_dir.mkdir(parents=True, exist_ok=True)

    generated_paths = []

    for i, text in enumerate(sentences, start=1):
        file_path = model_output_dir / f"{i:02d}_{model_name}.wav"

        model.tts_to_file(
            text=text,
            speaker_wav=str(speaker_wav),
            language=language,
            file_path=str(file_path),
        )

        generated_paths.append(file_path)

    return generated_paths

In [24]:
# Референсная запись.
reference_wav = sorted((FT_ALL_ROOT / "wavs").glob("*.wav"))[0]
reference_wav_2 = sorted((FT_ALL_ROOT / "wavs").glob("*.wav"))[1]
reference_wav_3 = sorted((FT_ALL_ROOT / "wavs").glob("*.wav"))[2]

print("Reference wav:", reference_wav)
display(Audio(str(reference_wav)))
display(Audio(str(reference_wav_2)))
display(Audio(str(reference_wav_3)))

Reference wav: /kaggle/working/xtts_finetune_dmitri/dmitri_all/wavs/utt_0000.wav


In [27]:
# Zero-shot модель

clear_cuda()

zero_shot_tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(config.device)

zero_shot_paths = generate_10_samples(
    model=zero_shot_tts,
    model_name="zero_shot",
    speaker_wav=reference_wav,
    sentences=TEST_SENTENCES,
    output_dir=EVAL_OUTPUT_DIR,
    language="ru",
)

clear_cuda()

print("Zero-shot generated files:")
for p in zero_shot_paths:
    print(p)

Zero-shot generated files:
outputs/eval_4_1/zero_shot/01_zero_shot.wav
outputs/eval_4_1/zero_shot/02_zero_shot.wav
outputs/eval_4_1/zero_shot/03_zero_shot.wav
outputs/eval_4_1/zero_shot/04_zero_shot.wav
outputs/eval_4_1/zero_shot/05_zero_shot.wav
outputs/eval_4_1/zero_shot/06_zero_shot.wav
outputs/eval_4_1/zero_shot/07_zero_shot.wav
outputs/eval_4_1/zero_shot/08_zero_shot.wav
outputs/eval_4_1/zero_shot/09_zero_shot.wav
outputs/eval_4_1/zero_shot/10_zero_shot.wav


In [25]:
# Модель, дообученная на 1 минуте

clear_cuda()

KAGGLE_MODEL_DIR = "/kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1"

ft_1min_tts = load_finetuned_tts_from_kaggle(
    kaggle_model_dir=KAGGLE_MODEL_DIR,
    device=config.device,
)


ft_1min_paths = generate_10_samples(
    model=ft_1min_tts,
    model_name="fine_tuned_1min",
    speaker_wav=reference_wav,
    sentences=TEST_SENTENCES,
    output_dir=EVAL_OUTPUT_DIR,
    language="ru",
)

clear_cuda()

print("Fine-tuned 1 min generated files:")
for p in ft_1min_paths:
    print(p)

Found checkpoint: /kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1/best_model.pth
Found config: /kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1/config.json
Loading fine-tuned XTTS from Kaggle:
model dir: /kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1
checkpoint: /kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1/best_model.pth
config: /kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1/config.json
vocab: /kaggle/input/models/twogenies/xtts_dmitri_1min/keras/xtts/1/vocab.json
Fine-tuned 1 min generated files:
outputs/eval_4_1/fine_tuned_1min/01_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/02_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/03_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/04_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/05_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/06_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/07_fine_tuned_1min.wav
outputs/eval_4_1/fine_tuned_1min/08_

In [26]:
# Модель, дообученная на всех данных

clear_cuda()

KAGGLE_MODEL_DIR = "/kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1"

ft_all_tts = load_finetuned_tts_from_kaggle(
    kaggle_model_dir=KAGGLE_MODEL_DIR,
    device=config.device,
)


ft_all_paths = generate_10_samples(
    model=ft_all_tts,
    model_name="fine_tuned_all",
    speaker_wav=reference_wav,
    sentences=TEST_SENTENCES,
    output_dir=EVAL_OUTPUT_DIR,
    language="ru",
)

clear_cuda()

print("Fine-tuned all-data generated files:")
for p in ft_all_paths:
    print(p)

Found checkpoint: /kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1/best_model.pth
Found config: /kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1/config.json
Loading fine-tuned XTTS from Kaggle:
model dir: /kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1
checkpoint: /kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1/best_model.pth
config: /kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1/config.json
vocab: /kaggle/input/models/twogenies/xtts_dmitri_all/keras/xtts/1/vocab.json
Fine-tuned all-data generated files:
outputs/eval_4_1/fine_tuned_all/01_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/02_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/03_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/04_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/05_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/06_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/07_fine_tuned_all.wav
outputs/eval_4_1/fine_tuned_all/08_fine_tuned_all.wav

In [28]:
print("GT reference:")
display(Audio(str(reference_wav)))

print("Zero-shot:")
display(Audio(str(zero_shot_paths[0])))

print("Fine-tuned 1 min:")
display(Audio(str(ft_1min_paths[0])))

print("Fine-tuned all data:")
display(Audio(str(ft_all_paths[0])))

GT reference:


Zero-shot:


Fine-tuned 1 min:


Fine-tuned all data:


**4.2.** Для каждой из моделей оцените speaker similarity моделью [WavLM-sv](https://huggingface.co/microsoft/wavlm-base-sv), как мы делали на семинаре

!!! в качестве референсной записи выберите 3 различных варианта и подсчитайте similarity:

- этих записей друг с другом

- этих записей со всеми сгенерированными

#### Загрузка модели

In [30]:
model_name = "microsoft/wavlm-base-sv"

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = WavLMForXVector.from_pretrained(model_name).to(config.device)
model.eval()

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

WavLMForXVector(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMGroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x WavLMNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMNoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): WavLMEncoder(
    

### Эмбединги и метрики

In [37]:
def load_audio(path, target_sr=16000):
    wav, sr = torchaudio.load(path)

    # mono
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    # resample
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, sr, target_sr)

    return wav.squeeze(0)


@torch.no_grad()
def get_embedding(wav):
    inputs = feature_extractor(
        wav,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )

    input_values = inputs.input_values.to(config.device)

    emb = model(input_values).embeddings  # (1, dim)
    emb = F.normalize(emb, dim=-1)

    return emb.squeeze(0)

def cosine_sim(a, b):
    return torch.dot(a, b).item()

In [43]:
import itertools
import pandas as pd

def embed_file(path: str | Path) -> torch.Tensor:
    wav = load_audio(str(path), target_sr=16000)
    return get_embedding(wav)

def mean_similarity(ref_paths, gen_paths):
    ref_embs = [embed_file(p) for p in ref_paths]
    gen_embs = [embed_file(p) for p in gen_paths]

    values = []
    for i, ref_emb in enumerate(ref_embs, start=1):
        for j, gen_emb in enumerate(gen_embs, start=1):
            values.append({
                "ref": f"ref{i}",
                "gen": f"gen{j}",
                "similarity": cosine_sim(ref_emb, gen_emb),
            })

    return pd.DataFrame(values)

# reference-reference
ref_embs = [embed_file(p) for p in ref_paths]

print("Reference-reference similarity:")
for (i, a), (j, b) in itertools.combinations(enumerate(ref_embs, start=1), 2):
    print(f"ref{i} vs ref{j}: {cosine_sim(a, b):.4f}")

# generated similarity по всем 10 файлам
model_to_paths = {
    "Zero-shot": zero_shot_paths,
    "Fine-tuned 1 min": ft_1min_paths,
    "Fine-tuned all data": ft_all_paths,
}

summary = []

for model_name, paths in model_to_paths.items():
    df = mean_similarity(ref_paths, paths)
    summary.append({
        "model": model_name,
        "mean_similarity": df["similarity"].mean(),
        "std_similarity": df["similarity"].std(),
        "min_similarity": df["similarity"].min(),
        "max_similarity": df["similarity"].max(),
    })

    print("\n", model_name)
    display(df)

summary_df = pd.DataFrame(summary)
display(summary_df)

Reference-reference similarity:
ref1 vs ref2: 0.9155
ref1 vs ref3: 0.9095
ref2 vs ref3: 0.9721

 Zero-shot


,ref,gen,similarity
0,ref1,gen1,0.961409
1,ref1,gen2,0.954693
2,ref1,gen3,0.966233
3,ref1,gen4,0.965059
4,ref1,gen5,0.948464
5,ref1,gen6,0.941206
6,ref1,gen7,0.955592
7,ref1,gen8,0.926254
8,ref1,gen9,0.947356
9,ref1,gen10,0.943338



 Fine-tuned 1 min


,ref,gen,similarity
0,ref1,gen1,0.952214
1,ref1,gen2,0.951042
2,ref1,gen3,0.974856
3,ref1,gen4,0.967995
4,ref1,gen5,0.965349
5,ref1,gen6,0.933659
6,ref1,gen7,0.929288
7,ref1,gen8,0.929469
8,ref1,gen9,0.964006
9,ref1,gen10,0.939215



 Fine-tuned all data


,ref,gen,similarity
0,ref1,gen1,0.939832
1,ref1,gen2,0.945424
2,ref1,gen3,0.966600
3,ref1,gen4,0.953578
4,ref1,gen5,0.958863
5,ref1,gen6,0.947617
6,ref1,gen7,0.963701
7,ref1,gen8,0.938433
8,ref1,gen9,0.958456
9,ref1,gen10,0.919976


,model,mean_similarity,std_similarity,min_similarity,max_similarity
0,Zero-shot,0.927209,0.025925,0.869426,0.966233
1,Fine-tuned 1 min,0.931129,0.023086,0.866038,0.974856
2,Fine-tuned all data,0.938441,0.021717,0.893194,0.973451


В результате оценки speaker similarity с использованием модели WavLM было показано, что fine-tuning модели XTTS приводит к улучшению сходства с референсным голосом.

Zero-shot модель продемонстрировала среднее значение similarity 0.927, в то время как fine-tuning на 1 минуте данных увеличил его до 0.931, а обучение на полном датасете — до 0.938.

Кроме увеличения среднего значения, наблюдается снижение стандартного отклонения и улучшение минимального значения similarity, что указывает на более стабильную генерацию речи.

Таким образом, дообучение модели позволяет не только повысить среднее качество клонирования голоса, но и уменьшить количество неудачных генераций, особенно при использовании большего объёма данных.